# Van der Waals gas and Maxwell's construction

*thermodynamics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/thermodynamics/van_der_waals)

The van der Waals equation adds molecular attraction and size to the ideal gas, and suddenly a liquid–gas transition appears. A whole family of isotherms is one $(T, V)$ broadcast; below the critical temperature the unphysical loop is replaced by Maxwell's equal-area line, whose pressure is found by bisection on a batch of temperatures at once.

**Vector form**

$$\Big(p + \frac{a n^2}{V^2}\Big)(V - nb) = nRT,\qquad T_c = \frac{8a}{27Rb},\ V_c = 3nb,\ p_c = \frac{a}{27b^2},\qquad \int_{V_l}^{V_g} \big(p - p^*\big)\,dV = 0$$

**Index (tensor) form**

$$p_{tv} = \frac{nRT_t}{V_v - nb} - \frac{an^2}{V_v^2},\qquad \Big[nRT\ln(V - nb) + \frac{an^2}{V}\Big]_{V_{l,t}}^{V_{g,t}} = p^*_t\,(V_{g,t} - V_{l,t})$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $T_t$ | `T[:, None]` | (T, 1) | sweep of temperatures, as a column |
| $V_v$ | `V[None, :]` | (1, V) | volume grid, as a row |
| $p_{tv}$ | `pressure(V, T, n, a, b)` | (T, V) | every isotherm at once by broadcasting |
| $V_{l,t},\ V_{g,t}$ | `eigvals(companion).real.sort()` | (B, 3) | roots of the cubic $p(V) = p^*$ for a batch of temperatures |
| $\int (p - p^*)\,dV$ | `area_mismatch(p_star, T, ...)` | (B,) | signed area between curve and line: zero at the answer |
| $p^*_t$ | `coexistence_pressure(T, ...)` | (B,) | bisection: torch.where moves every bracket at once |

$a$ measures attraction between molecules (it lowers the pressure) and $b$ the volume one mole of molecules takes up ($V$ cannot go below $nb$). Below $T_c$ the isotherm has a section where compressing the gas would lower its pressure — mechanically unstable. In reality the gas condenses at constant pressure $p^*$, the vapour pressure: liquid of molar volume $V_l$ coexists with gas of volume $V_g$. Maxwell's rule fixes $p^*$ by requiring equal areas above and below the line (equivalently, equal Gibbs energies of the two phases). The dashed dome joins all $(V_l, p^*)$ and $(V_g, p^*)$; its top is the critical point, where liquid and gas become identical. Axes are scaled by the critical values, in which every van der Waals gas looks the same (law of corresponding states); the volume axis is linear so the shaded areas can be compared by eye. The van der Waals model is only qualitative for real fluids: e.g. its $V_c = 3nb$ overestimates real critical volumes.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/thermodynamics/van_der_waals/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    n = params["n"]
    a, b = tensors.GASES[params["gas"]]
    T_c, V_c, p_c = equation.critical_point(n, a, b)
    T = tensors.temperatures(T_c, params["T_high"], params["T_low"], params["n_temps"])  # (T,)
    V = tensors.volumes(V_c)  # (V,)

    r = solve.solve(T, V, T_c, V_c, n, a, b)
    print(f"critical point: T_c = {T_c:.1f} K   V_c = {1e6 * V_c:.1f} cm³   p_c = {1e-6 * p_c:.3f} MPa")
    for t in [len(T) // 2, 3 * len(T) // 4, len(T) - 1]:
        if T[t] < T_c:
            print(
                f"T = {T[t]:6.1f} K   p* = {1e-6 * r['p_star'][t]:.3f} MPa"
                f"   liquid V = {1e6 * r['V_l'][t]:6.1f} cm³   gas V = {1e6 * r['V_g'][t]:7.1f} cm³"
            )
    return plot.animate(T, V, r, (T_c, V_c, p_c), out_dir / "van_der_waals.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/thermodynamics/van_der_waals/equation.py
r"""Physics → tensor translation.

The van der Waals equation corrects the ideal gas for molecular attraction (a) and molecular size (b):

    \Big(p + \frac{a n^2}{V^2}\Big)(V - n b) = n R T     ⇔     p = \frac{nRT}{V - nb} - \frac{a n^2}{V^2}

For a family of isotherms T_t sampled at volumes V_v (no index summed — pure broadcasting):

    p_{tv} = \frac{n R T_t}{V_v - n b} - \frac{a n^2}{V_v^2}

The critical point is where an isotherm has a flat inflection, ∂p/∂V = ∂²p/∂V² = 0:

    T_c = \frac{8a}{27Rb},     V_c = 3 n b,     p_c = \frac{a}{27 b^2}

Below T_c the isotherm has an unphysical loop. Maxwell's equal-area rule replaces it by a flat line at the
coexistence pressure p^*, chosen so the areas above and below the line cancel:

    \int_{V_l}^{V_g} (p(V) - p^*) \, dV = 0,      where p(V_l) = p(V_g) = p^*

The integral of p is known in closed form: ∫p dV = nRT ln(V - nb) + a n² / V. The volumes where p(V) = p^*
are the roots of the cubic p^* V³ − (p^* n b + nRT) V² + a n² V − a n³ b = 0, found for every temperature
at once as eigenvalues of its companion matrix. p^* itself comes from bisection on a batch of temperatures.
"""

import torch

R = 8.314  # gas constant (J / (mol K))


def pressure(V: torch.Tensor, T: torch.Tensor, n: float, a: float, b: float) -> torch.Tensor:
    """p_{tv} = nRT_t / (V_v − nb) − a n² / V_v².   V: (V,), T: (T,) → (T, V)"""
    return n * R * T[:, None] / (V[None, :] - n * b) - a * n**2 / V[None, :] ** 2


def critical_point(n: float, a: float, b: float) -> tuple[float, float, float]:
    """(T_c, V_c, p_c) = (8a / 27Rb, 3nb, a / 27b²)."""
    return 8 * a / (27 * R * b), 3 * n * b, a / (27 * b**2)


def volume_roots(p: torch.Tensor, T: torch.Tensor, n: float, a: float, b: float) -> torch.Tensor:
    """The three volumes where the isotherm crosses pressure p, sorted.   p, T: (B,) → (B, 3)

    Divide the cubic by p to get V³ + c₂V² + c₁V + c₀ = 0. Its companion matrix has the roots as eigenvalues.
    """
    c2 = -(n * b + n * R * T / p)
    c1 = a * n**2 / p
    c0 = -a * n**3 * b / p
    companion = torch.zeros(len(p), 3, 3, dtype=p.dtype, device=p.device)  # (B, 3, 3)
    companion[:, 0] = torch.stack([-c2, -c1, -c0], dim=-1)
    companion[:, 1, 0] = 1.0
    companion[:, 2, 1] = 1.0
    return torch.linalg.eigvals(companion).real.sort(dim=-1).values  # three real roots inside the loop


def work_integral(V1: torch.Tensor, V2: torch.Tensor, T: torch.Tensor, n: float, a: float, b: float) -> torch.Tensor:
    """∫ p dV from V1 to V2 along an isotherm = nRT ln((V2 − nb) / (V1 − nb)) + a n² (1/V2 − 1/V1)."""
    return n * R * T * torch.log((V2 - n * b) / (V1 - n * b)) + a * n**2 * (1 / V2 - 1 / V1)


def area_mismatch(p_star: torch.Tensor, T: torch.Tensor, n: float, a: float, b: float) -> torch.Tensor:
    """∫_{V_l}^{V_g} (p − p*) dV: positive when the line p* sits too low.   (B,) → (B,)"""
    roots = volume_roots(p_star, T, n, a, b)
    V_l, V_g = roots[:, 0], roots[:, 2]  # liquid and gas ends of the flat line
    return work_integral(V_l, V_g, T, n, a, b) - p_star * (V_g - V_l)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/thermodynamics/van_der_waals/tensors.py
"""Build the concrete tensors: gas constants, the sweep of temperatures and the volume grid (float64)."""

import math

import torch

# van der Waals constants: a in Pa m⁶ / mol², b in m³ / mol
GASES = {
    "CO2": (0.3640, 4.267e-5),
    "water": (0.5536, 3.049e-5),
    "nitrogen": (0.1370, 3.870e-5),
    "argon": (0.1355, 3.201e-5),
}


def temperatures(T_c: float, high: float, low: float, n: int) -> torch.Tensor:
    """T_t falling from high × T_c to low × T_c.  Shape (T,)."""
    return T_c * torch.linspace(high, low, n, dtype=torch.float64)


def volumes(V_c: float, points: int = 1500) -> torch.Tensor:
    """V_v from 0.4 V_c to 8 V_c, logarithmically spaced (the gas side is long and flat).  Shape (V,)."""
    return V_c * torch.logspace(math.log10(0.4), math.log10(8.0), points, dtype=torch.float64)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/thermodynamics/van_der_waals/solve.py
"""Isotherms for every temperature, plus Maxwell's equal-area line for those below T_c (bisection on tensors)."""

import torch

from .equation import area_mismatch, pressure, volume_roots


def coexistence_pressure(
    T: torch.Tensor, V: torch.Tensor, V_c: float, n: float, a: float, b: float, n_iter: int = 60
) -> torch.Tensor:
    """Bisection for p* with ∫(p − p*) dV = 0, all temperatures (all below T_c) at once.  (B,) → (B,)

    The answer lies between the bottom of the loop (local minimum, at V < V_c) and its top (local maximum,
    at V > V_c). Each iteration halves every bracket in one tensor operation.
    """
    p = pressure(V, T, n, a, b)  # (B, V)
    low = torch.where(V < V_c, p, torch.inf).min(dim=-1).values  # (B,)  bottom of the loop
    high = torch.where(V > V_c, p, -torch.inf).max(dim=-1).values  # (B,)  top of the loop
    low = low.clamp(min=1e-3 * high)  # at low T the loop dips below p = 0; p* is always positive
    for _ in range(n_iter):
        mid = 0.5 * (low + high)
        too_low = area_mismatch(mid, T, n, a, b) > 0  # more area above the line than below → raise it
        low = torch.where(too_low, mid, low)
        high = torch.where(too_low, high, mid)
    return 0.5 * (low + high)


def solve(T: torch.Tensor, V: torch.Tensor, T_c: float, V_c: float, n: float, a: float, b: float) -> dict:
    """Return isotherms p (T, V) and, for T < T_c, the flat line: p* and its ends V_l, V_g (T,) (NaN above T_c)."""
    p = pressure(V, T, n, a, b)  # (T, V)
    below = T < T_c  # (T,)
    p_star = torch.full_like(T, float("nan"))
    V_l, V_g = p_star.clone(), p_star.clone()

    if below.any():
        p_star[below] = coexistence_pressure(T[below], V, V_c, n, a, b)
        roots = volume_roots(p_star[below], T[below], n, a, b)  # (B, 3)
        V_l[below], V_g[below] = roots[:, 0], roots[:, 2]
    return {"p": p, "p_star": p_star, "V_l": V_l, "V_g": V_g}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/thermodynamics/van_der_waals/plot.py
"""Animate the isotherm as T falls through T_c: the loop, its equal areas, the flat line and the growing dome."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif


def animate(T: torch.Tensor, V: torch.Tensor, r: dict, critical: tuple, path: Path) -> Path:
    T_c, V_c, p_c = critical
    temps = T.numpy()
    v = V.numpy() / V_c  # (V,)  reduced volume V / V_c
    p = r["p"].numpy() / p_c  # (T, V)  reduced pressure p / p_c
    p_star = r["p_star"].numpy() / p_c  # (T,)  NaN above T_c
    v_l, v_g = r["V_l"].numpy() / V_c, r["V_g"].numpy() / V_c  # (T,)

    fig, (ax, ax_pT) = plt.subplots(1, 2, figsize=(7, 4), gridspec_kw={"width_ratios": [1.6, 1]})

    # Left: p–V plane with the current isotherm, its Maxwell line and the coexistence dome so far.
    for t in range(0, len(temps), max(1, len(temps) // 8)):
        ax.plot(v, p[t], color="#d1d5db", lw=0.8)  # faint reference isotherms
    curve = ax.plot([], [], color=ACCENT[0], lw=2)[0]
    flat = ax.plot([], [], color="black", lw=1.5)[0]
    dome = ax.plot([], [], color=ACCENT[3], lw=1.5, ls="--", label="coexistence dome")[0]
    ax.plot([1], [1], "*", color=ACCENT[1], ms=10, label="critical point")
    shading = []
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_xlim(v[0], 5.5)  # linear axis, so the two shaded areas really look equal
    ax.set_ylim(min(0.0, p[:, v < 1].min()) - 0.05, 1.6)  # the loop can dip below p = 0
    ax.set_xlabel("V / V_c")
    ax.set_ylabel("p / p_c")
    ax.legend(fontsize=8, loc="upper right")
    title = ax.set_title("")

    # Right: vapour-pressure curve p*(T), ending at the critical point.
    ax_pT.plot([1], [1], "*", color=ACCENT[1], ms=10)
    vapour = ax_pT.plot([], [], color=ACCENT[3], lw=2)[0]
    ax_pT.set_xlim(temps.min() / T_c, temps.max() / T_c)
    ax_pT.set_ylim(0, 1.15)
    ax_pT.set_xlabel("T / T_c")
    ax_pT.set_ylabel("p* / p_c")
    ax_pT.set_title("liquid–gas coexistence", fontsize=10)

    def update(step: int):
        for patch in shading:
            patch.remove()
        shading.clear()
        curve.set_data(v, p[step])
        done = ~np.isnan(p_star[: step + 1])  # temperatures below T_c reached so far
        if not np.isnan(p_star[step]):
            ps = p_star[step]
            inside = (v >= v_l[step]) & (v <= v_g[step])
            flat.set_data([v_l[step], v_g[step]], [ps, ps])
            shading.append(ax.fill_between(v, p[step], ps, where=inside & (p[step] < ps), color=ACCENT[0], alpha=0.3))
            shading.append(ax.fill_between(v, p[step], ps, where=inside & (p[step] > ps), color=ACCENT[1], alpha=0.3))
        else:
            flat.set_data([], [])
        # dome: liquid ends from low T up to the critical point, then gas ends back down
        dome_v = np.concatenate([v_l[: step + 1][done][::-1], [1.0], v_g[: step + 1][done]])
        dome_p = np.concatenate([p_star[: step + 1][done][::-1], [1.0], p_star[: step + 1][done]])
        dome.set_data(dome_v, dome_p)
        vapour.set_data(np.append(1.0, temps[: step + 1][done] / T_c), np.append(1.0, p_star[: step + 1][done]))
        title.set_text(f"T = {temps[step]:.0f} K = {temps[step] / T_c:.2f} T_c   (equal shaded areas)")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(temps)), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'gas': 'CO2',
    'n': 1.0,
    'T_high': 1.15,
    'T_low': 0.8,
    'n_temps': 90
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.thermodynamics.van_der_waals")]:
    del sys.modules[name]
main = importlib.import_module("problems.thermodynamics.van_der_waals.main")
problem_dir = Path("problems/thermodynamics/van_der_waals")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
